# XGBoost Training Pipeline V2 — Prompt Injection Detection

**Dataset:** 534k samples, 403 features (19 hand-crafted + 384 embeddings), balanced binary classification
**Model:** XGBoost GPU (CUDA)
**Optimization:** Optuna 100 trials, 5-fold CV, AUC-ROC maximization

Run cells sequentially. Cell 6 (Optuna) takes ~30-60 min on GPU.

In [ ]:
import subprocess, sys
for pkg in ['xgboost>=2.0', 'optuna>=3.5', 'shap>=0.44']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna
import shap
import json, pickle, os, time, warnings
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve,
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print('XGBoost: ', xgb.__version__)
print('Optuna:  ', optuna.__version__)
print('SHAP:    ', shap.__version__)

try:
    _dm = xgb.DMatrix([[1, 2], [3, 4]], label=[0, 1])
    xgb.train({'device': 'cuda', 'tree_method': 'hist', 'eval_metric': 'auc'},
              _dm, num_boost_round=1, verbose_eval=False)
    print('OK: XGBoost GPU (cuda) working')
except Exception as e:
    print('WARN: GPU unavailable:', e)
    print('  Set device="cpu" in CONFIG (Cell 2)')

In [ ]:
CONFIG = {
    'data_path': 'data/features_engineered_v2.csv',
    'original_data_path': 'data/prompt_injection_dataset.csv',
    'models_dir': 'models',
    'test_size': 0.20,
    'cv_folds': 5,
    'seed': 42,
    'n_trials': 100,
    'study_name': 'xgboost_prompt_injection_v2',
    'device': 'cuda',
    'tree_method': 'hist',
    'eval_metric': 'auc',
    'early_stopping_rounds': 50,
    'hp': {
        'n_estimators':     (300, 3000),
        'max_depth':        (3, 10),
        'learning_rate':    (0.005, 0.3),
        'subsample':        (0.5, 1.0),
        'colsample_bytree': (0.3, 1.0),
        'min_child_weight': (1, 15),
        'gamma':            (1e-8, 5.0),
        'reg_alpha':        (1e-8, 10.0),
        'reg_lambda':       (1e-8, 10.0),
    },
}
os.makedirs(CONFIG['models_dir'], exist_ok=True)
print(f"Models directory ready: {CONFIG['models_dir']}")
print(f"Device: {CONFIG['device']} | Trials: {CONFIG['n_trials']}")

In [ ]:
df = pd.read_csv(CONFIG['data_path'])
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Expected: 404 columns (403 features + label)')
assert df.shape[1] == 404, f'Expected 404 columns, got {df.shape[1]}'
assert 'label' in df.columns

median_ld = df['lexical_diversity'].median()
df['lexical_diversity'] = df['lexical_diversity'].fillna(median_ld)
nulls = df.isnull().sum()
remaining = nulls[nulls > 0]
if len(remaining) > 0:
    print("WARNING: Remaining nulls:")
    print(remaining)
    df = df.fillna(0)
print(f'Zero nulls after preprocessing')
print(f'\nClass distribution:')
print(df['label'].value_counts().to_string())
print(df['label'].value_counts(normalize=True).round(4).to_string())

In [ ]:
FEATURE_COLS = [c for c in df.columns if c != 'label']
X = df[FEATURE_COLS].reset_index(drop=True)
y = df['label'].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG['test_size'], stratify=y, random_state=CONFIG['seed']
)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f'Train: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test:  {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'Features: {len(FEATURE_COLS)}')
assert abs(y_train.mean() - y_test.mean()) < 0.01, 'Stratification failed'
print('Stratification check passed')

In [ ]:
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators',     *CONFIG['hp']['n_estimators']),
        'max_depth':        trial.suggest_int('max_depth',        *CONFIG['hp']['max_depth']),
        'learning_rate':    trial.suggest_float('learning_rate',   *CONFIG['hp']['learning_rate'], log=True),
        'subsample':        trial.suggest_float('subsample',       *CONFIG['hp']['subsample']),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *CONFIG['hp']['colsample_bytree']),
        'min_child_weight': trial.suggest_int('min_child_weight',  *CONFIG['hp']['min_child_weight']),
        'gamma':            trial.suggest_float('gamma',           *CONFIG['hp']['gamma'], log=True),
        'reg_alpha':        trial.suggest_float('reg_alpha',       *CONFIG['hp']['reg_alpha'], log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda',      *CONFIG['hp']['reg_lambda'], log=True),
        'device':       CONFIG['device'],
        'tree_method':  CONFIG['tree_method'],
        'eval_metric':  CONFIG['eval_metric'],
        'random_state': CONFIG['seed'],
        'verbosity':    0,
        'early_stopping_rounds': CONFIG['early_stopping_rounds'],
    }
    cv = StratifiedKFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=CONFIG['seed'])
    fold_aucs = []
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_train.iloc[tr_idx], y_train.iloc[tr_idx],
            eval_set=[(X_train.iloc[val_idx], y_train.iloc[val_idx])],
            verbose=False,
        )
        proba = model.predict_proba(X_train.iloc[val_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y_train.iloc[val_idx], proba))
        trial.report(np.mean(fold_aucs), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(fold_aucs))

print('Optuna objective defined')
print(f'  CV folds: {CONFIG["cv_folds"]}, early stopping: {CONFIG["early_stopping_rounds"]}')
print('  Primary metric: AUC-ROC (maximize)')

In [ ]:
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=CONFIG['seed']),
    pruner=optuna.pruners.MedianPruner(),
    study_name=CONFIG['study_name'],
)
study.optimize(objective, n_trials=CONFIG['n_trials'], show_progress_bar=True)

print(f'\nBest AUC-ROC: {study.best_value:.6f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
best_params = study.best_params.copy()
best_params.update({
    'device': CONFIG['device'],
    'tree_method': CONFIG['tree_method'],
    'eval_metric': CONFIG['eval_metric'],
    'random_state': CONFIG['seed'],
    'verbosity': 0,
    'early_stopping_rounds': CONFIG['early_stopping_rounds'],
})

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=CONFIG['seed']
)

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=100,
)
print(f'\nBest iteration: {final_model.best_iteration}')
print(f'Total trees: {final_model.n_estimators}')

In [ ]:
y_proba = final_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba > 0.5).astype(int)

auc_roc = roc_auc_score(y_test, y_proba)
auc_pr = average_precision_score(y_test, y_proba)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'\n=== V2 TEST RESULTS ===')
print(f'AUC-ROC:  {auc_roc:.4f}')
print(f'AUC-PR:   {auc_pr:.4f}')
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1 Score:  {f1:.4f}')
print(f'\nV1 Baseline: AUC-ROC=0.9180, Recall=0.7724, FN rate=22.8%')

cm = confusion_matrix(y_test, y_pred)
print(f'\nConfusion Matrix:')
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')
print(f'  FP rate: {cm[0,1]/(cm[0,0]+cm[0,1]):.1%}')
print(f'  FN rate: {cm[1,0]/(cm[1,0]+cm[1,1]):.1%}')

In [ ]:
importance_gain = final_model.get_booster().get_score(importance_type='gain')
importance_weight = final_model.get_booster().get_score(importance_type='weight')

hc_importance = {k: v for k, v in importance_gain.items() if not k.startswith('embedding_')}
hc_importance = dict(sorted(hc_importance.items(), key=lambda x: -x[1]))

print("=== Top Hand-Crafted Features (Gain) ===")
for i, (feat, score) in enumerate(list(hc_importance.items())[:19], 1):
    print(f"  {i:2d}. {feat:30s} {score:.4f}")

In [ ]:
model_path = os.path.join(CONFIG['models_dir'], 'xgboost_model_v2.json')
final_model.get_booster().save_model(model_path)
print(f"Model saved to {model_path}")
print(f"  Size: {os.path.getsize(model_path) / 1024**2:.1f} MB")

params_path = os.path.join(CONFIG['models_dir'], 'best_params_v2.json')
with open(params_path, 'w') as f:
    json.dump(study.best_params, f, indent=2)
print(f"Best params saved to {params_path}")

metrics = {
    'auc_roc': float(auc_roc),
    'auc_pr': float(auc_pr),
    'accuracy': float(acc),
    'precision': float(prec),
    'recall': float(rec),
    'f1': float(f1),
    'fn_rate': float(cm[1,0] / (cm[1,0] + cm[1,1])),
    'fp_rate': float(cm[0,1] / (cm[0,0] + cm[0,1])),
    'n_features': len(FEATURE_COLS),
    'n_handcrafted': 19,
    'n_embeddings': 384,
    'best_iteration': int(final_model.best_iteration),
    'v1_baseline_auc': 0.9180,
    'v1_baseline_fn_rate': 0.228,
}
metrics_path = os.path.join(CONFIG['models_dir'], 'evaluation_metrics_v2.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")

study_path = os.path.join(CONFIG['models_dir'], 'optuna_study_v2.pkl')
with open(study_path, 'wb') as f:
    pickle.dump(study, f)
print(f"Optuna study saved to {study_path}")